In [1]:
import os
import duckdb
from typing import Self, Literal

# GCSの承認

In [2]:
GCS_SA_KEY_PATH = os.environ.get("GCS_SA_KEY_PATH", r"/workspace/configuration/keita-masui-firstproject-46adeb7731b9.json")

# DuckDBクラスの定義

In [3]:
STAGING_DB_PATH = r"/workspace/src/stg/duck_db/staging.duckdb"

In [4]:
class DuckDBConnector:
    """A class that manages the connection to DuckDB and functions as a context manager.
    """
    def __init__(self, db_type:Literal["staging"]):
        """_summary_.
        Args:
            db_type (Literal[&quot;staging&quot;]): _description_
        """
        if db_type == "staging":
            self.db_path = STAGING_DB_PATH
        else:
            raise ValueError("Invalid db_type")

        self.conn = None

    def __enter__(self) -> duckdb.DuckDBPyConnection:
        """

        Returns:
            duckdb.DuckDBPyConnection: _description_
        """
        if not os.path.exists(os.path.dirname(self.db_path)):
            os.makedirs(os.path.dirname(self.db_path), exist_ok=True)

        # 接続を確立
        self.conn = duckdb.connect(database=self.db_path)

        # S3/GCSの連携
        self._setup_cloud_connection()

        return self.conn

    def __exit__(self, exc_type, exc_val, exc_tb):
        """_summary_.

        Args:
            exc_type (_type_): _description_
            exc_val (_type_): _description_
            exc_tb (_type_): _description_
        """
        if self.conn:
            self.conn.close()

    def _setup_cloud_connection(self) -> None:
        """_summary_.
        """
        if GCS_SA_KEY_PATH:
            self.conn.execute(f"""
                INSTALL httpfs;
                LOAD httpfs;
                SET gcs_service_account_key='{GCS_SA_KEY_PATH}';
                SET gcs_credential_type = 'service_account';
            """)

# GCS Bucketへの接続

In [ ]:
conn.execute(f"""
             """)